In [1]:
# Implémentation de l'élagage L1 non structuré en Python avec PyTorch
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

# Définir un modèle simple
model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 5)
)

# Sélectionner la couche à élaguer
layer_to_prune = model[0]
param_to_prune = 'weight'

# Appliquer un élagage non structuré de 50% basé sur la magnitude L1
prune.l1_unstructured(layer_to_prune, name=param_to_prune, amount=0.5)

# Le tenseur de poids est maintenant un produit du masque et des poids d'origine
print(f"Poids élagués:\n{layer_to_prune.weight}")
print(f"Masque d'élagage:\n{layer_to_prune.weight_mask}")

# Pour rendre l'élagage permanent et supprimer le masque
prune.remove(layer_to_prune, param_to_prune)
print(f"Poids permanents après élagage:\n{layer_to_prune.weight}")

Poids élagués:
tensor([[ 0.0000, -0.2570, -0.2584,  0.2905,  0.2777, -0.0000,  0.1579, -0.0000,
         -0.0000, -0.1567],
        [ 0.2303,  0.0000, -0.2658, -0.0000, -0.0000,  0.2405,  0.0000, -0.2590,
         -0.0000, -0.0000],
        [-0.0000, -0.2473, -0.0000, -0.2539,  0.1683,  0.0000, -0.0000,  0.2320,
         -0.1872, -0.0000],
        [ 0.2282, -0.0000, -0.2517,  0.0000, -0.0000,  0.1709,  0.0000,  0.0000,
         -0.2762, -0.2024],
        [-0.0000,  0.0000,  0.0000,  0.2844,  0.0000,  0.0000,  0.0000,  0.0000,
          0.3060, -0.1573],
        [-0.2352,  0.2403,  0.0000,  0.2067, -0.1582, -0.0000, -0.0000,  0.1691,
          0.2581, -0.0000],
        [ 0.2473, -0.0000, -0.2263,  0.0000,  0.0000, -0.1620,  0.0000,  0.0000,
          0.2783, -0.0000],
        [-0.3129, -0.3149, -0.0000, -0.2351,  0.2071,  0.2325,  0.0000, -0.0000,
          0.2694, -0.0000],
        [ 0.0000,  0.2559,  0.2930, -0.0000, -0.2411,  0.0000,  0.1927,  0.0000,
          0.0000, -0.2744],
    

In [3]:
import torch.nn as nn
from torch.ao.quantization import QuantStub, DeQuantStub, get_default_qconfig, prepare_qat, convert

# 1. Définir un modèle compatible avec la quantification
class QuantizableModel(nn.Module):
    def __init__(self):
        super(QuantizableModel, self).__init__()
        self.quant = QuantStub()  # Convertit FP32 -> INT8
        self.conv = nn.Conv2d(1, 1, 1)
        self.relu = nn.ReLU()
        self.dequant = DeQuantStub() # Convertit INT8 -> FP32

    def forward(self, x):
        x = self.quant(x)
        x = self.conv(x)
        x = self.relu(x)
        x = self.dequant(x)
        return x

# 2. Préparer le modèle pour la quantification
model_fp32 = QuantizableModel()
model_fp32.eval()
model_fp32.qconfig = get_default_qconfig('fbgemm')
model_fp32_prepared = torch.ao.quantization.prepare(model_fp32)

# 3. Calibrer avec des données représentatives
input_fp32 = torch.randn(4, 1, 4, 4)
model_fp32_prepared(input_fp32)

# 4. Convertir en modèle quantifié
model_int8 = torch.ao.quantization.convert(model_fp32_prepared)

# L'inférence se fait maintenant avec des opérations entières
output = model_int8(input_fp32)
print("Modèle quantifié exécuté avec succès.")

Modèle quantifié exécuté avec succès.


In [5]:
import torch
import torch.nn as nn
import torch.quantization

# 1. Définir un modèle FP32 compatible avec la quantification
class SimpleQuantModel(nn.Module):
    def __init__(self):
        super(SimpleQuantModel, self).__init__()
        # QuantStub et DeQuantStub marquent les points d'entrée/sortie de la section quantifiée
        self.quant = torch.quantization.QuantStub()
        self.conv = nn.Conv2d(1, 1, 3)
        self.relu = nn.ReLU()
        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.conv(x)
        x = self.relu(x)
        x = self.dequant(x)
        return x

# Créer une instance du modèle
model_fp32 = SimpleQuantModel()
model_fp32.eval()

# 2. Préparer le modèle pour la quantification
model_fp32.qconfig = torch.quantization.get_default_qconfig('fbgemm') # Backend pour x86
model_fp32_fused = torch.quantization.fuse_modules(model_fp32, [['conv', 'relu']])
model_fp32_prepared = torch.quantization.prepare(model_fp32_fused)

# 3. Calibrer le modèle avec des données représentatives
# Dans un cas réel, on utiliserait un DataLoader avec des données de validation
calibration_data = [torch.randn(1, 1, 28, 28) for _ in range(100)]
with torch.no_grad():
    for data in calibration_data:
        model_fp32_prepared(data)

# 4. Convertir le modèle en INT8
model_int8 = torch.quantization.convert(model_fp32_prepared)

# L'inférence se fait maintenant avec des opérations INT8
input_fp32 = torch.randn(1, 1, 28, 28)
output = model_int8(input_fp32)
print("Modèle INT8 exécuté avec succès.")

Modèle INT8 exécuté avec succès.


In [4]:
import torch.nn as nn
import torch.nn.functional as F

def distillation_loss(student_logits, teacher_logits, hard_labels, temp, alpha):
    # Perte sur les cibles molles (soft targets)
    soft_loss = nn.KLDivLoss(reduction='batchmean')(
        F.log_softmax(student_logits / temp, dim=1),
        F.softmax(teacher_logits / temp, dim=1)
    ) * (temp * temp)

    # Perte sur les cibles dures (hard targets)
    hard_loss = F.cross_entropy(student_logits, hard_labels)

    # Combinaison des deux pertes
    total_loss = alpha * hard_loss + (1. - alpha) * soft_loss
    return total_loss

# Dans la boucle d'entraînement...
# teacher_model.eval()
# student_outputs = student_model(inputs)
# with torch.no_grad():
#     teacher_outputs = teacher_model(inputs)
#
# loss = distillation_loss(student_outputs, teacher_outputs, labels, temp=4.0, alpha=0.3)
# loss.backward()
# optimizer.step()

In [7]:
import torch
import torch.nn as nn

# Un bloc OFA conceptuel qui supporte une largeur variable
class OFABlock(nn.Module):
    def __init__(self, max_in_channels, max_out_channels):
        super(OFABlock, self).__init__()
        self.conv = nn.Conv2d(max_in_channels, max_out_channels, 3, padding=1)
        self.active_out_channels = max_out_channels

    def forward(self, x):
        # Utiliser seulement les canaux d'entrée actifs
        active_in_channels = x.shape[1]
        # Appliquer la convolution avec les poids complets
        full_output = self.conv(active_in_channels)
        # Retourner seulement les canaux de sortie actifs
        return full_output[:, :self.active_out_channels, :, :]

# Le super-réseau
class SuperNet(nn.Module):
    def __init__(self):
        super(SuperNet, self).__init__()
        self.block1 = OFABlock(3, 64)
        self.block2 = OFABlock(64, 128)
    
    def forward(self, x):
        return self.block2(self.block1(x))

# Créer le super-réseau
supernet = SuperNet()
# Entraînement du super-réseau (omis)

# Spécialiser pour un sous-réseau
def specialize_subnet(supernet_model, config):
    supernet_model.block1.active_out_channels = config['block1_width']
    # Le nombre de canaux d'entrée de block2 dépend de la sortie de block1
    # En pratique, on utilise des couches linéaires pour adapter les dimensions
    # ou on s'assure que les largeurs sont compatibles.
    # Ici, nous allons simplement tronquer les poids d'entrée de block2.
    
    # Note: Ceci est une simplification. L'OFA réel utilise des techniques plus sophistiquées.
    # La convolution de block2 utilisera tous ses poids d'entrée, mais seulement les
    # `config['block1_width']` premiers canaux de l'activation d'entrée seront non nuls.
    
    supernet_model.block2.active_out_channels = config['block2_width']
    return supernet_model

# Config pour un petit sous-réseau
small_config = {'block1_width': 32, 'block2_width': 64}
small_net = specialize_subnet(supernet, small_config)

# Inférence avec le petit réseau
input_tensor = torch.randn(1, 3, 32, 32)
output = small_net(input_tensor)
print(f"Sortie du petit réseau: {output.shape}") # Devrait être [1, 64, 32, 32]

TypeError: conv2d() received an invalid combination of arguments - got (int, Parameter, Parameter, tuple, tuple, tuple, int), but expected one of:
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, tuple of ints padding = 0, tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!int!, !Parameter!, !Parameter!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, !int!)
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, str padding = "valid", tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!int!, !Parameter!, !Parameter!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, !int!)


In [8]:
import torch
import torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super(TransformerBlock, self).__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Multi-Head Attention
        attn_output, _ = self.attention(x, x, x)
        # Add & Norm (connexion résiduelle)
        x = self.norm1(x + self.dropout(attn_output))
        # Feed Forward
        ffn_output = self.ffn(x)
        # Add & Norm
        x = self.norm2(x + self.dropout(ffn_output))
        return x

# Exemple d'utilisation
block = TransformerBlock(embed_dim=512, num_heads=8, ff_dim=2048)
input_seq = torch.randn(10, 32, 512) # (seq_len, batch_size, embed_dim)
output_seq = block(input_seq)
print(output_seq.shape)

torch.Size([10, 32, 512])


In [9]:
import torch

def smoothquant_transform(x, w, alpha=0.5):
    """Applique la transformation SmoothQuant à un tenseur d'activation et de poids."""
    # x: [num_tokens, in_channels], w: [in_channels, out_channels]
    
    # Calculer les échelles par canal
    act_scales = torch.max(torch.abs(x), dim=0)[0]
    weight_scales = torch.max(torch.abs(w), dim=0)[0]
    
    # Calculer le facteur de lissage 's'
    s = act_scales.pow(alpha) / weight_scales.pow(1 - alpha)
    s = torch.clamp(s, min=1e-5) # Éviter la division par zéro

    # Appliquer la transformation
    x_hat = x / s
    w_hat = w * s
    
    return x_hat, w_hat

# Exemple
in_channels, out_channels, num_tokens = 128, 256, 50
x = torch.randn(num_tokens, in_channels)
w = torch.randn(in_channels, out_channels)

# Introduire des outliers dans les activations
x[:, 10] *= 100 
x[:, 20] *= 200

x_hat, w_hat = smoothquant_transform(x, w)

print(f"Max abs original X: {torch.max(torch.abs(x[:, 10]))}, {torch.max(torch.abs(x[:, 20]))}")
print(f"Max abs lissé X_hat: {torch.max(torch.abs(x_hat[:, 10]))}, {torch.max(torch.abs(x_hat[:, 20]))}")
# On observera que les valeurs maximales dans x_hat sont réduites.

RuntimeError: The size of tensor a (128) must match the size of tensor b (256) at non-singleton dimension 0